## Data Engineering


In [6]:
import pandas as pd

# Load the final filtered dataset
review_data = pd.read_json('../data/reviews_5core_sample_filtered.jsonl', lines=True)
metadata = pd.read_json('../data/metadata_5core_sample.jsonl', lines=True)

print(review_data.columns)
print(metadata.columns)


AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

In [ ]:
print(review_data.isna().sum())
print(metadata.isna().sum())

metadata['main_category'] = metadata['main_category'].fillna("Unknown")
metadata.loc[(metadata['main_category'] == "nan") | (metadata['main_category'] == 'NaN'), 'main_category'] = "Unknown"

metadata['price'] = pd.to_numeric(metadata['price'], errors='coerce')
metadata['price'] = metadata['price'].fillna(metadata['price'].median())
metadata['bought_together'] = metadata['bought_together'].apply(lambda x: x if isinstance(x, list) else [])

metadata = metadata.drop(columns=['subtitle', 'author', 'store'])



#Apply 5core filtering

user_counts = review_data['user_id'].value_counts()
item_counts = review_data['parent_asin'].value_counts()

# review_data = review_data[
#     review_data['user_id'].isin(user_counts[user_counts >= 5].index) &
#     review_data['parent_asin'].isin(item_counts[item_counts >= 5].index)
# ]

print(review_data.isna().sum())
print(metadata.isna().sum())

In [ ]:
# Check for duplicates
print("Before deduplication:")
print(f"Total reviews: {len(review_data)}")
duplicates = review_data.duplicated(subset=['user_id', 'parent_asin'])
print(f"Duplicate (user, item) pairs: {duplicates.sum()}")

# Remove duplicates - keep latest timestamp and average rating for duplicates
if duplicates.sum() > 0:
    # Group by user and item, average the rating, keep latest timestamp
    review_data = review_data.groupby(['user_id', 'parent_asin']).agg({
    'rating': 'mean',
    'timestamp': 'max',
    'verified_purchase': 'last',
    'helpful_vote': 'last',
    'text': 'last',        # Keep text from most recent review
    'title': 'last'        # Keep title from most recent review
}).reset_index()




In [ ]:
# Step 1: Convert to datetime
review_data['reviewTime'] = pd.to_datetime(review_data['timestamp'], unit='s')

# Step 2: Sort by time (useful for train/test splitting)
review_data = review_data.sort_values(by='reviewTime')

# Step 3: Normalize timestamps
min_time = review_data['reviewTime'].min()
review_data['days_since_start'] = (review_data['reviewTime'] - min_time).dt.days

# # Save the processed data
# review_data.to_json('../data/processed/review_data.jsonl', orient='records', lines=True)
# metadata.to_json('../data/processed/metadata.jsonl', orient='records', lines=True)

# print("Saved processed data to ../data/processed")

In [4]:
import matplotlib.pyplot as plt

user_counts = review_data['user_id'].value_counts()
item_counts = review_data['parent_asin'].value_counts()
#Summary statistics
print("Number of unique users:", review_data['user_id'].nunique())
print("Number of unique items:", review_data['parent_asin'].nunique())
print("Total number of reviews:", len(review_data))
print("User review counts:", user_counts.describe())
print("Item review counts:", item_counts.describe())

# Histograms 
plt.hist(user_counts, bins=50, log=True)
plt.title("Distribution of reviews per user")
plt.xlabel("# Reviews per user")
plt.ylabel("Count of users")
plt.show()

plt.hist(item_counts, bins=50, log=True)
plt.title("Distribution of reviews per item")
plt.xlabel("# Reviews per item")
plt.ylabel("Count of items")
plt.show()




ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
popular_items_data = review_data['parent_asin'].value_counts().reset_index()
popular_items_data.columns = ['parent_asin', 'review_count']

popular_items_with_name = pd.merge(
    popular_items_data,
    metadata[['title', 'parent_asin']],
    on = 'parent_asin',
    how='left'
)

print("\nTop 10 items with review counts:")
print(popular_items_with_name.head(10))

import pandas as pd

# Count reviews per item
item_counts = review_data['parent_asin'].value_counts()

# Total number of items
n_items = len(item_counts)

# Total reviews
total_reviews = item_counts.sum()

# Top 1%, 5%, 10%
for pct in [0.01, 0.05, 0.10]:
    top_k = int(n_items * pct)
    share = item_counts.head(top_k).sum() / total_reviews * 100
    print(f"Top {int(pct*100)}% of items cover {share:.2f}% of all reviews")


In [ ]:
#Extract year month from the review time
review_data['year_month'] = review_data['reviewTime'].dt.to_period('M')

# Count reviews per month
monthly_reviews = review_data['year_month'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(monthly_reviews.index.astype(str), monthly_reviews.values, marker='o')
plt.title("Number of reviews per month")
plt.xlabel('Month')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print some statistics
print(f"Total months covered: {len(monthly_reviews)}")
print(f"Average reviews per month: {monthly_reviews.mean():.1f}")
print(f"Peak month: {monthly_reviews.idxmax()} with {monthly_reviews.max()} reviews")


# Temporal distribution
review_data['year'] = review_data['reviewTime'].dt.year
review_data['year'].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Year")
plt.ylabel("Number of reviews")
plt.show()

#### Interpretations
- Since all the years dont have similar number of reviews, doing a train test split based on year or timestamp will not work without accounting for this bias.
- Less than 2010, data is sparse and can be ignored or downweighted for modelling.
- The majority of items will have less number of reviews, need to adjust for popularity of few items. The data shows a skew towards amazon basics products.
- The top 10% of items cover 42% of reviews. The popular items dominate the review set. Need to downweight the popularity bias. Any model would overfit to these items, need to mitigate it.

